In [7]:
import json
import random

from datasets import load_dataset

# Load MedMCQA dataset
medmcqa = load_dataset("openlifescienceai/medmcqa")

print("MedMCQA splits:", medmcqa)

MedMCQA splits: DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 182822
    })
    test: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 6150
    })
    validation: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 4183
    })
})


In [8]:
# Check the column names and a sample
print(medmcqa["train"].column_names)
print(medmcqa["train"][0])

['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name']
{'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': 2, 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}


In [9]:
# Filter for single choice questions only with explanations
# choice_type: "single" or "multi"
train_single = medmcqa["train"].filter(lambda x: x["choice_type"] == "single" and x["exp"])
test_single = medmcqa["validation"].filter(lambda x: x["choice_type"] == "single")

print(f"Train single choice with explanation: {len(train_single)}")
print(f"Test single choice: {len(test_single)}")

Train single choice with explanation: 106370
Test single choice: 2816


In [10]:
# Set seed for reproducibility
random.seed(42)

# Sample 3000 for train and 1000 for test
train_indices = random.sample(range(len(train_single)), 3000)
test_indices = random.sample(range(len(test_single)), 1000)

train_sampled = train_single.select(train_indices)
test_sampled = test_single.select(test_indices)

print(f"Sampled train: {len(train_sampled)}")
print(f"Sampled test: {len(test_sampled)}")

Sampled train: 3000
Sampled test: 1000


In [11]:
# Transform the dataset to the required format
def transform_medmcqa(dataset):
    transformed = []
    option_ids = ["A", "B", "C", "D"]

    for item in dataset:
        options = [item["opa"], item["opb"], item["opc"], item["opd"]]
        correct_idx = item["cop"]  # 0-indexed correct option
        answer_id = option_ids[correct_idx]

        transformed.append(
            {
                "id": item["id"],
                "question": item["question"],
                "explanation": item["exp"],
                "options": options,
                "option_ids": option_ids,
                "answer": answer_id,
            }
        )
    return transformed


train_transformed = transform_medmcqa(train_sampled)
test_transformed = transform_medmcqa(test_sampled)

print(f"Transformed train: {len(train_transformed)}")
print(f"Transformed test: {len(test_transformed)}")
print("\nSample:", train_transformed[0])

Transformed train: 3000
Transformed test: 1000

Sample: {'id': '04563ab2-d5cd-4e0a-94a3-d680a274d151', 'question': 'Burn by moist heat is known as:', 'explanation': 'Burns may be classified as:\n\nOrdinary burns caused by dry heat.\nScalds caused by moist heat.\nChemical burns caused by strong acid or base.\nElectric burns.\nRadiation bums.\nCold burns.', 'options': ['Ordinary burn', 'Scar burn', 'Scalds burn', 'Hot burn'], 'option_ids': ['A', 'B', 'C', 'D'], 'answer': 'C'}


In [12]:
import os

# Save to JSONL
train_output_path = "../../data/source/medmcqa/medmcqa_train.jsonl"
test_output_path = "../../data/source/medmcqa/medmcqa_test.jsonl"

os.makedirs(os.path.dirname(train_output_path), exist_ok=True)

with open(train_output_path, "w") as f:
    for item in train_transformed:
        f.write(json.dumps(item) + "\n")

with open(test_output_path, "w") as f:
    for item in test_transformed:
        f.write(json.dumps(item) + "\n")

print(f"Saved {len(train_transformed)} samples to {train_output_path}")
print(f"Saved {len(test_transformed)} samples to {test_output_path}")

Saved 3000 samples to ../../data/source/medmcqa/medmcqa_train.jsonl
Saved 1000 samples to ../../data/source/medmcqa/medmcqa_test.jsonl
